# Lecture 5: Identification -- Which Parameters Can Summaries See?

Everything run in class today lives in this file (Python) and its R counterpart. The hands-on lab (Parts A and B) is in a separate file, `05-identifiability-lab.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt

rng = np.random.default_rng(681)  # set once, here -- never inside a simulator function

def norm_ppf(p, lo=-10.0, hi=10.0):
    # standard normal quantile function, via bisection on erf (no scipy needed)
    def cdf(x):
        return 0.5 * (1 + erf(x / sqrt(2)))
    for _ in range(100):
        mid = (lo + hi) / 2
        if cdf(mid) < p:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


# Section 1: A repair -- the Normal distribution

## Fixed $\sigma$, varying $\mu$

In [ ]:
n_draws = 4000
mus = [-2, 0, 2]
sigma_fixed = 1

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2), sharex=True, sharey=True)
for ax, m in zip(axes, mus):
    ax.hist(rng.normal(loc=m, scale=sigma_fixed, size=n_draws), bins=40, color="steelblue")
    ax.set_title(f"mu = {m}")
    ax.set_xlim(-6, 6)
axes[0].set_ylabel("count")
fig.suptitle("Fixed sigma = 1, varying mu")
plt.tight_layout()
plt.show()


## Fixed $\mu$, varying $\sigma$

In [ ]:
sigmas = [0.5, 1, 2]
mu_fixed = 0

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2), sharex=True, sharey=True)
for ax, s in zip(axes, sigmas):
    ax.hist(rng.normal(loc=mu_fixed, scale=s, size=n_draws), bins=40, color="darkorange")
    ax.set_title(f"sigma = {s}")
    ax.set_xlim(-6, 6)
axes[0].set_ylabel("count")
fig.suptitle("Fixed mu = 0, varying sigma")
plt.tight_layout()
plt.show()


## The location-scale construction

In [ ]:
z = rng.normal(size=n_draws)  # Z ~ N(0,1)
mu, sigma = 3, 2
y = mu + sigma * z  # Y ~ N(mu, sigma^2)

print("mean_z:", z.mean(), " sd_z:", z.std())
print("mean_y:", y.mean(), " sd_y:", y.std())


`mu` shifts the standard normal draws; `sigma` stretches them. This is literally what `rng.normal(loc=mu, scale=sigma, size=n)` does internally.

# Section 4: Building the identification surfaces

We use the *population* (expected) values of these summaries, computed analytically, rather than simulated averages -- finite-simulation noise can produce a spurious minimum in sigma (a good lesson for later, but not today's point).

For $Y\sim N(\mu,\sigma^2)$: $E(Y)=\mu$, and $q_p = \mu + \sigma z_p$ where $z_p = \Phi^{-1}(p)$.

In [ ]:
def m_mean(mu, sigma):
    return mu

def m_quant(mu, sigma, p):
    return mu + sigma * norm_ppf(p)

print("z_50:", norm_ppf(0.50), " z_60:", norm_ppf(0.60), " z_75:", norm_ppf(0.75))


## An observed dataset

In [ ]:
mu0, sigma0, n_obs = 5, 2, 50
y_obs = rng.normal(loc=mu0, scale=sigma0, size=n_obs)

obs_mean = y_obs.mean()
obs_median = np.median(y_obs)
obs_p60 = np.quantile(y_obs, 0.60)
obs_p75 = np.quantile(y_obs, 0.75)

print("mean:", obs_mean, " median:", obs_median, " p60:", obs_p60, " p75:", obs_p75)


## The three objective surfaces

In [ ]:
mu_grid = np.linspace(0, 10, 220)
sigma_grid = np.linspace(0.1, 6, 220)
Mu, Sigma = np.meshgrid(mu_grid, sigma_grid)

def objective(p2, obs2):
    return (obs_mean - m_mean(Mu, Sigma))**2 + (obs2 - m_quant(Mu, Sigma, p2))**2

Q_median = objective(0.50, obs_median)
Q_p60 = objective(0.60, obs_p60)
Q_p75 = objective(0.75, obs_p75)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.1), sharex=True, sharey=True)
titles = ["mean + median (nonidentified)", "mean + 60th pct. (weakly identified)",
          "mean + 75th pct. (identified)"]
for ax, Q, title in zip(axes, [Q_median, Q_p60, Q_p75], titles):
    cs = ax.contourf(Mu, Sigma, np.sqrt(Q), levels=20, cmap="viridis")
    ax.contour(Mu, Sigma, np.sqrt(Q), levels=15, colors="white", alpha=0.5, linewidths=0.5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("mu")
axes[0].set_ylabel("sigma")
plt.tight_layout()
plt.show()


The left panel is a perfectly vertical valley: $Q$ genuinely does not depend on $\sigma$ once $\mu$ is right. The middle panel's valley is visibly curved, but still much less sharply localized than the right panel's genuine minimum.

This all worked because we happen to know $m(\mu,\sigma)$ for the Normal exactly: the population $p$-quantile is $\mu + \sigma z_p$, so sliding $p$ from $0.5$ to $0.60$ to $0.75$ is really just sliding $z_p$ from $0$ up to $0.67$ -- one number, directly controlling how tight the valley is. That closed form is special to the Normal. For the SIR model below, there is no such formula for $m(\beta,\gamma)$; simulation is the only way to see the corresponding geometry.

# Section 5: Transfer to the SIR model

In [ ]:
def simulate_sir(n_steps, beta, gamma, S0, I0, R0, rng):
    N = S0 + I0 + R0
    S = np.zeros(n_steps + 1, dtype=int)
    I = np.zeros(n_steps + 1, dtype=int)
    R = np.zeros(n_steps + 1, dtype=int)
    S[0], I[0], R[0] = S0, I0, R0
    for t in range(n_steps):
        p_infect = 1 - (1 - beta / N) ** I[t]
        delta_I = rng.binomial(S[t], p_infect)
        delta_R = rng.binomial(I[t], gamma)
        S[t + 1] = S[t] - delta_I
        I[t + 1] = I[t] + delta_I - delta_R
        R[t + 1] = R[t] + delta_R
    return S, I, R

N = 1000
I0 = 20
n_steps = 450
window = 8

def sir_summary_once(beta, gamma, rng):
    S, I, R = simulate_sir(n_steps, beta, gamma, N - I0, I0, 0, rng)
    final_size = N - S[-1]
    I_early = I[: window + 1]
    t_early = np.arange(window + 1)
    growth_rate = np.polyfit(t_early, np.log(np.maximum(I_early, 1)), 1)[0]
    return np.array([final_size, growth_rate])

def sir_summary_avg(beta, gamma, B, rng):
    draws = np.array([sir_summary_once(beta, gamma, rng) for _ in range(B)])
    return draws.mean(axis=0)


## Checking the "same $R_0$" and "same $\beta-\gamma$" claims

In [ ]:
B_check = 300
same_R0_a = sir_summary_avg(0.30, 0.10, B_check, rng)  # R0 = 3, beta-gamma = 0.20
same_R0_b = sir_summary_avg(0.60, 0.20, B_check, rng)  # R0 = 3, beta-gamma = 0.40
same_diff = sir_summary_avg(0.40, 0.20, B_check, rng)  # R0 = 2, beta-gamma = 0.20

print("(0.30, 0.10), R0=3:", same_R0_a)
print("(0.60, 0.20), R0=3:", same_R0_b)
print("(0.40, 0.20), R0=2, same beta-gamma as row 1:", same_diff)


Rows 1 and 2 share $R_0=3$: their final sizes are nearly identical, but their growth rates are quite different. Rows 1 and 3 share $\beta-\gamma=0.20$: their growth rates are close, but their final sizes are quite different.

## The grid computation

This takes a little while -- 256 grid points, each averaged over 120 simulated epidemics.

In [ ]:
beta_grid = np.linspace(0.10, 1.00, 16)
gamma_grid = np.linspace(0.05, 0.50, 16)

B_grid = 120
final_size_grid = np.zeros((len(gamma_grid), len(beta_grid)))
growth_rate_grid = np.zeros((len(gamma_grid), len(beta_grid)))

for i, g in enumerate(gamma_grid):
    for j, b in enumerate(beta_grid):
        avg = sir_summary_avg(b, g, B_grid, rng)
        final_size_grid[i, j] = avg[0]
        growth_rate_grid[i, j] = avg[1]


## The final-size surface

In [ ]:
Beta, Gamma = np.meshgrid(beta_grid, gamma_grid)

fig, ax = plt.subplots(figsize=(6.5, 5.2))
cs = ax.contourf(Beta, Gamma, final_size_grid, levels=20, cmap="viridis")
ax.contour(Beta, Gamma, final_size_grid, levels=12, colors="white", alpha=0.6, linewidths=0.6)
for R0 in [1, 2, 4, 6, 8]:
    ax.plot(beta_grid, beta_grid / R0, color="red", linestyle="--")
ax.set_xlim(beta_grid.min(), beta_grid.max())
ax.set_ylim(gamma_grid.min(), gamma_grid.max())
ax.set_xlabel("beta")
ax.set_ylabel("gamma")
ax.set_title("Final size depends mainly on R0 = beta / gamma")
fig.colorbar(cs, label="final size")
plt.tight_layout()
plt.show()


## The growth-rate surface

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.2))
cs = ax.contourf(Beta, Gamma, growth_rate_grid, levels=20, cmap="viridis")
ax.contour(Beta, Gamma, growth_rate_grid, levels=12, colors="white", alpha=0.6, linewidths=0.6)
for d in [0.1, 0.3, 0.5, 0.7, 0.9]:
    ax.plot(beta_grid, beta_grid - d, color="red", linestyle="--")
ax.set_xlim(beta_grid.min(), beta_grid.max())
ax.set_ylim(gamma_grid.min(), gamma_grid.max())
ax.set_xlabel("beta")
ax.set_ylabel("gamma")
ax.set_title("Early growth rate depends mainly on beta - gamma")
fig.colorbar(cs, label="growth rate")
plt.tight_layout()
plt.show()


The white contours in the two plots run in different directions: the first tracks the red $R_0=$ const lines, the second tracks the red $\beta-\gamma=$ const lines. This is what $m(\beta,\gamma)$ itself looks like -- not yet an SMM objective built from one fixed observed dataset. Let's build that now, live, the same way we built the Normal objective surfaces earlier: pick one fixed "observed" epidemic, and compare it to the grid we just simulated.

## Building the actual objective surface, live

In [ ]:
beta0, gamma0 = 0.30, 0.10  # the true parameters, not used below except to draw obs
obs = sir_summary_once(beta0, gamma0, rng)
obs_final_size, obs_growth_rate = obs
print(obs)


In [ ]:
B_var = 300
reps = np.array([sir_summary_once(beta0, gamma0, rng) for _ in range(B_var)])
var_final = reps[:, 0].var()
var_growth = reps[:, 1].var()
print("var_final:", var_final, " var_growth:", var_growth)


In [ ]:
Q_final = (obs_final_size - final_size_grid) ** 2
Q_growth = (obs_growth_rate - growth_rate_grid) ** 2
Q_naive = Q_final + Q_growth
Q_std = Q_final / var_final + Q_growth / var_growth

def plot_Q(Q, title, k=100):
    # Clip the color scale to (min(Q), k * min(Q)): what matters for judging
    # identifiability is how much of the grid is close to the best value
    # found, not Q's full raw range -- and final size's raw range is many
    # orders of magnitude larger than growth rate's, which would otherwise
    # swamp any comparison.
    Qmin = Q.min()
    fig, ax = plt.subplots(figsize=(6, 5))
    cs = ax.pcolormesh(Beta, Gamma, Q, cmap="viridis", vmin=Qmin, vmax=k * Qmin, shading="auto")
    ax.contour(Beta, Gamma, Q, levels=12, colors="white", alpha=0.5, linewidths=0.5)
    ax.plot(beta0, gamma0, marker="x", color="red", markersize=10, markeredgewidth=2)
    ax.set_xlabel("beta")
    ax.set_ylabel("gamma")
    ax.set_title(title)
    fig.colorbar(cs, label="Q", extend="max")
    plt.tight_layout()
    plt.show()

plot_Q(Q_final, "final size alone")
plot_Q(Q_growth, "growth rate alone")
plot_Q(Q_naive, "both, unweighted")
plot_Q(Q_std, "both, standardized")


In [ ]:
def argmin_beta_gamma(Q):
    idx = np.unravel_index(np.argmin(Q), Q.shape)
    return beta_grid[idx[1]], gamma_grid[idx[0]]

print("true: beta =", beta0, " gamma =", gamma0)
print("final_alone: ", argmin_beta_gamma(Q_final))
print("growth_alone:", argmin_beta_gamma(Q_growth))
print("naive:       ", argmin_beta_gamma(Q_naive))
print("standardized:", argmin_beta_gamma(Q_std))


Each panel's color scale is clipped to (min $Q$, 100$\times$ min $Q$): what matters for judging identifiability is how much of the grid is *close to the best value found*, not $Q$'s full raw range -- and final size's raw range is many orders of magnitude larger than growth rate's, which would otherwise swamp any comparison. The red $\times$ marks the true $(\beta,\gamma)$ on every panel.

With that fixed, the two ridges are visible right where the earlier $m(\beta,\gamma)$ surfaces said they'd be: final size's near-minimum band runs the length of the $R_0=3$ diagonal, growth rate's runs along a *different* diagonal ($\beta-\gamma=$ const). Both bands do pass through the true point -- that's reassuring, and it has to be true, since the true parameters really do produce a small residual on each summary alone. But neither band is any *darker* at the true point than anywhere else along it: final size doesn't distinguish the true $(\beta,\gamma)$ from any other point sharing its $R_0$, and growth rate doesn't distinguish it from any other point sharing its $\beta-\gamma$. Naive (unweighted) combination lands right on top of final-size-alone, exactly like "first attempt: add squared discrepancies" did in Lecture 4. Standardizing is the one panel whose minimum is genuinely a *point*, not a stretch of a line -- the darkest cell sits right at the true parameters, and $Q$ climbs as you move away in either direction along the diagonal. That's the real signature of identifiability: not "this region is colorful," but "the color changes as you leave the truth."

# Section 6: The exponential distribution

In [ ]:
lambdas = [0.5, 1, 2]

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2), sharex=True, sharey=True)
for ax, l in zip(axes, lambdas):
    ax.hist(rng.exponential(scale=1 / l, size=n_draws), bins=40, color="seagreen")
    ax.set_title(f"lambda = {l}")
    ax.set_xlim(0, 8)
axes[0].set_ylabel("count")
fig.suptitle("Standard exponential (lambda = 1) and varying lambda")
plt.tight_layout()
plt.show()


$\lambda = 1$ is the *standard* exponential. Larger $\lambda$ concentrates more mass near zero -- events happen more often, so waits are shorter, on average $1/\lambda$. Smaller $\lambda$ spreads the distribution out further to the right.